In [1]:
#Load cleaned files 

import pandas as pd 
import matplotlib.pyplot as plt 

patients = pd.read_csv(r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\cleaned\clean_patients.csv")
visits   = pd.read_csv(r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\cleaned\clean_visits.csv")
claims   = pd.read_csv(r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\cleaned\clean_claims.csv")
followups= pd.read_csv(r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\cleaned\clean_followups.csv")
print("Files loaded successfully")

Files loaded successfully


In [8]:
patient_features = visits.groupby(
    "patient_id"
).agg(
    visit_frequency=("visit_id", "count"),
    total_spend=("treatment_cost", "sum")
).reset_index()

# ✅ Round total_spend to 2 decimal places
patient_features["total_spend"] = patient_features["total_spend"].round(2)

print(patient_features.head())

   patient_id  visit_frequency  total_spend
0           1                6     28737.23
1           2                4     32734.61
2           3                4     44624.12
3           4                7     39944.65
4           5                2     18831.72


In [9]:
#Readmission count

readmission_counts = followups[
    followups["readmission_flag"] == "Yes"
].groupby(
    "patient_id"
).size().reset_index(
    name="readmission_count"
)

print(readmission_counts.head()) 

   patient_id  readmission_count
0           2                  1
1           3                  1
2           4                  2
3           6                  2
4           7                  1


In [10]:
#Merge Features

patient_features = patient_features.merge(
    readmission_counts,
    on="patient_id",
    how="left"
)

patient_features[
    "readmission_count"
] = patient_features[
    "readmission_count"
].fillna(0)

print(patient_features.head())

   patient_id  visit_frequency  total_spend  readmission_count
0           1                6     28737.23                0.0
1           2                4     32734.61                1.0
2           3                4     44624.12                1.0
3           4                7     39944.65                2.0
4           5                2     18831.72                0.0


In [11]:
#Create risk score

patient_features["risk_score"] = (
    patient_features["visit_frequency"] * 2
    +
    patient_features["readmission_count"] * 5
)

In [12]:
#risk segmentation

def segment_risk(score):

    if score >= 30:
        return "Critical"

    elif score >= 20:
        return "High Risk"

    elif score >= 10:
        return "Moderate"

    else:
        return "Low Risk"


patient_features[
    "patient_segment"
] = patient_features[
    "risk_score"
].apply(
    segment_risk
)

print(
    patient_features[
        "patient_segment"
    ].value_counts()
)

patient_segment
Moderate     25918
Low Risk     16464
High Risk     6209
Critical       507
Name: count, dtype: int64


In [13]:
#save features

patient_features.to_csv(
    r"D:\python project files\Health Care Revenue Cycle & Patient Risk Analytics\data\processed\patient_features.csv",
    index=False
)

print("patient_features.csv created successfully")

patient_features.csv created successfully
